In [3]:

# 1. Import libraries
import pandas as pd
import re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 2. Load dataset
df = pd.read_csv("Arabic_Quotes.csv")

# Change this to your actual quote column name
text_col = "Quote"

# 3. Arabic text cleaning
def clean_arabic(text):
    text = str(text)
    text = re.sub(r'[إأآا]', 'ا', text)       # Normalize Alef
    text = re.sub(r'ى', 'ي', text)            # Normalize Yaa
    text = re.sub(r'ة', 'ه', text)            # Normalize Taa Marbuta
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)  # Remove diacritics
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)     # Keep Arabic
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = df.dropna(subset=[text_col])
df["clean"] = df[text_col].apply(clean_arabic)

# 4. TF-IDF
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df["clean"])

# 5. K-Means
k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["Cluster"] = model.fit_predict(X)

# 6. Show clusters
for i in range(k):
    print("\nCLUSTER", i)
    print(df[df["Cluster"] == i][text_col].head(5).to_string(index=False))

# 7. Visualize clusters
pca = PCA(n_components=2, random_state=42)
points = pca.fit_transform(X.toarray())

plt.figure(figsize=(8, 6))
plt.scatter(points[:, 0], points[:, 1], c=df["Cluster"], cmap="viridis")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Arabic Quotes - K-Means Clustering")
plt.colorbar(label="Cluster")
plt.show()

# 8. Save results
df.to_csv("Arabic_Quotes_Clusters.csv", index=False, encoding="utf-8-sig")



KeyError: ['Quote']